# WHAM extract -> normalized SMPL `.npz`

Runs [WHAM](https://github.com/yohanshin/WHAM) (world-grounded SMPL) on one monocular
clip of a single person and writes `<video_id>.wham.npz`, which
`tools/smpl_to_skeleton.py --wham-output` turns into `skeleton.json v2`.

- Design: `docs/superpowers/specs/2026-07-23-monocular-smpl-skeleton-design.md`
- Output contract (npz keys): see `tools/colab/README.md` in this same folder.

**One-time setup:** Runtime -> Change runtime type -> GPU. Have your neutral SMPL
model ready to upload: locally it is `models/smpl/SMPL_NEUTRAL.pkl` (see the README
there). Then run the cells top to bottom.


In [ ]:
!git clone https://github.com/yohanshin/WHAM.git
%cd WHAM
!bash install.sh                 # WHAM's own setup (torch, deps, checkpoints)
!pip install smplx loguru        # smplx = joint regressor; loguru is imported by demo.py but install.sh sometimes misses it on Colab


In [ ]:
from google.colab import files
print("Upload SMPL_NEUTRAL.pkl (from smpl.is.tue.mpg.de)")
up = files.upload()
import os, shutil
os.makedirs("body_models/smpl", exist_ok=True)
shutil.copy(next(iter(up)), "body_models/smpl/SMPL_NEUTRAL.pkl")


In [ ]:
VIDEO_ID = "test_6"                      # <-- name of this clip (test_6 = the Pexels person-moving clip)
up = files.upload()                      # pick your .mp4 (e.g. test_6.mp4)
clip = next(iter(up))

# WHAM does not need 4K. Downscale to 720p (keep aspect, even dims; fps unchanged)
# so detection/tracking runs fast. Remove this if the clip is already small.
clip720 = "input_720.mp4"
!ffmpeg -y -i "{clip}" -vf "scale=-2:720" -an "{clip720}"

!python demo.py --video "{clip720}" --output_pth output/{VIDEO_ID} --save_pkl --visualize


In [ ]:
import numpy as np, torch, joblib, glob, smplx

pkls = sorted(glob.glob(f"output/{VIDEO_ID}/*.pkl"))
assert pkls, (f"No WHAM output .pkl under output/{VIDEO_ID}/. "
              "Cell 3 (demo.py) errored before finishing — scroll up for the real cause "
              "(e.g. a missing module) and fix that first, then re-run Cell 3.")
res = joblib.load(pkls[0])
track = res[sorted(res.keys())[0]]     # first tracked person

# WHAM stores world-grounded params when available.
pose  = np.asarray(track.get("pose_world", track["pose"]), dtype=np.float32)   # (T,72)
transl = np.asarray(track.get("trans_world", track["trans"]), dtype=np.float32) # (T,3)
betas = np.asarray(track["betas"], dtype=np.float32)
if betas.ndim == 2:
    betas = betas.mean(0)              # (10,)

body = smplx.create("body_models", model_type="smpl", gender="neutral", batch_size=pose.shape[0])
out = body(global_orient=torch.tensor(pose[:, :3]),
           body_pose=torch.tensor(pose[:, 3:72]),
           betas=torch.tensor(betas[None].repeat(pose.shape[0], 0)),
           transl=torch.tensor(transl))
joints3d = out.joints.detach().cpu().numpy()[:, :24, :]   # (T,24,3)

# fps from the source video
import cv2
fps = cv2.VideoCapture(clip).get(cv2.CAP_PROP_FPS) or 30.0

np.savez(f"{VIDEO_ID}.wham.npz", joints3d=joints3d, pose=pose,
         betas=betas, transl=transl, fps=np.array(fps))
print("shapes:", joints3d.shape, pose.shape, betas.shape, transl.shape, "fps", fps)
files.download(f"{VIDEO_ID}.wham.npz")
